In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
!pip install gradio opencv-python tensorflow

In [4]:
import os

dataset_path = "/kaggle/input/trashnet/dataset-resized"

In [5]:
import cv2
import numpy as np
from tensorflow.keras.utils import to_categorical

classes = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

images = []
labels = []

IMG_SIZE = 128

for label, category in enumerate(classes):

    folder_path = os.path.join("C:\\Users\\STUDENTS\\Desktop\\TrashNet\\dataset-resized", category)

    for file in os.listdir(folder_path):

        img_path = os.path.join(folder_path, file)

        img = cv2.imread(img_path)

        # Skip broken images
        if img is None:
            continue

        # Resize image
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

        # Convert BGR to RGB
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Normalize pixels
        img = img / 255.0

        images.append(img)
        labels.append(label)

X = np.array(images)
y = to_categorical(labels)

print("Dataset Loaded Successfully!")
print(X.shape)
print(y.shape)  

Dataset Loaded Successfully!
(2527, 128, 128, 3)
(2527, 6)


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import Flatten, Dense, Dropout

model = Sequential([

    Conv2D(32, (3,3), activation='relu',
           input_shape=(128,128,3)),

    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),

    Dense(128, activation='relu'),

    Dropout(0.5),

    Dense(6, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

c:\Users\STUDENTS\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,305,414 (12.61 MB)

 Trainable params: 3,305,414 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    validation_data=(X_test, y_test)
)

Epoch 1/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 18s 255ms/step - accuracy: 0.3271 - loss: 1.6098 - val_accuracy: 0.4170 - val_loss: 1.4310
Epoch 2/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 14s 220ms/step - accuracy: 0.4448 - loss: 1.3649 - val_accuracy: 0.4684 - val_loss: 1.2989
Epoch 3/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 13s 209ms/step - accuracy: 0.5022 - loss: 1.2435 - val_accuracy: 0.4387 - val_loss: 1.4689
Epoch 4/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 13s 202ms/step - accuracy: 0.5210 - loss: 1.1982 - val_accuracy: 0.5711 - val_loss: 1.1603
Epoch 5/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 13s 210ms/step - accuracy: 0.5685 - loss: 1.1095 - val_accuracy: 0.5968 - val_loss: 1.0950


In [9]:
model.save("trash_classifier.h5")

In [10]:
def predict_image(image):

    img = cv2.resize(image, (128,128))

    img = img / 255.0

    img = np.expand_dims(img, axis=0)

    prediction = model.predict(img)

    predicted_class = classes[np.argmax(prediction)]

    confidence = np.max(prediction)

    return f"{predicted_class} ({confidence*100:.2f}%)"

In [11]:
import gradio as gr
import numpy as np

# Prediction function
def predict_image(image):
    # Example prediction logic
    return "Plastic Waste"   # Replace with your AI model prediction

# Gradio Interface
interface = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="numpy"),
    outputs="text",
    title="AI Trash Classifier",
    description="Upload trash image and AI will classify it"
)

interface.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
